# CLOAQ demo

Obfuscate a QASMBench circuit, pass it through an untrusted compiler, and restore it with the correct key. Run `pip install -e ".[dev]"` first.

In [1]:
import random
import sys

import numpy as np
from qiskit import qasm2, transpile
from qiskit_aer import AerSimulator

sys.path.append('../benchmarks')
from run_benchmarks import fetch  # downloads QASMBench circuits on demand

from cloaq import deobfuscate, functionally_equivalent, obfuscate, simulate, tvd, wrong_key

qc = qasm2.load(str(fetch('adder_n4')), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
ref = simulate(qc, seed=1)
print('original output:', ref)
qc.draw('text', fold=120)

original output: {'1001': 4096}


┌───┐┌───┐            ┌───┐          ┌─────┐                ┌───┐     ┌─┐   
q_0: ┤ X ├┤ T ├───■────────┤ X ├───────■──┤ Tdg ├──■─────────────┤ X ├─────┤M├───
     ├───┤├───┤ ┌─┴─┐      └─┬─┘     ┌─┴─┐├─────┤┌─┴─┐     ┌─┐   └─┬─┘     └╥┘   
q_1: ┤ X ├┤ T ├─┤ X ├────────┼────■──┤ X ├┤ Tdg ├┤ X ├─────┤M├─────┼────────╫────
     └───┘└───┘ ├───┤        │  ┌─┴─┐└───┘├─────┤└───┘     └╥┘┌─┐  │        ║    
q_2: ───────■───┤ T ├───■────┼──┤ X ├──■──┤ Tdg ├──■────────╫─┤M├──┼────────╫────
     ┌───┐┌─┴─┐┌┴───┴┐┌─┴─┐  │  └───┘┌─┴─┐└┬───┬┘┌─┴─┐┌───┐ ║ └╥┘  │  ┌───┐ ║ ┌─┐
q_3: ┤ H ├┤ X ├┤ Tdg ├┤ X ├──■───────┤ X ├─┤ T ├─┤ X ├┤ S ├─╫──╫───■──┤ H ├─╫─┤M├
     └───┘└───┘└─────┘└───┘          └───┘ └───┘ └───┘└───┘ ║  ║      └───┘ ║ └╥┘
c: 4/═══════════════════════════════════════════════════════╩══╩════════════╩══╩═
                                                            1  2            0  3

## Obfuscate → untrusted compile → de-obfuscate

In [2]:
circuit, logic_key, phase_key = obfuscate(qc, seed=42)
print('logic key :', logic_key)
print('phase bits:', len(phase_key))
print('ops (obfuscated):', dict(circuit.count_ops()))

compiled = transpile(circuit, AerSimulator(), optimization_level=0)
restored = deobfuscate(compiled, logic_key, phase_key)

print('ops (restored)  :', dict(restored.count_ops()))
print('exactly equivalent:', functionally_equivalent(restored, qc))
print('TVD correct key   :', round(tvd(ref, simulate(restored, seed=1)), 4))

logic key : [1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0]
phase bits: 108
ops (obfuscated): {'barrier': 94, 'p': 36, 'h': 13, 'cx': 11, 'ccx': 5, 'measure': 4, 'x': 1}
ops (restored)  : {'cx': 10, 'p': 9, 'measure': 4, 'x': 2, 'h': 2}
exactly equivalent: True
TVD correct key   : 0.0


## Wrong keys

In [3]:
rng = random.Random(0)
cases = [('logic only', True, False), ('phase only', False, True), ('both', True, True)]
for label, bl, bp in cases:
    vals = [tvd(ref, simulate(deobfuscate(compiled,
                wrong_key(logic_key, rng) if bl else logic_key,
                wrong_key(phase_key, rng) if bp else phase_key), seed=1)) for _ in range(20)]
    print(f'wrong {label:<11}: mean TVD = {np.mean(vals):.3f}')

wrong logic only : mean TVD = 0.936


wrong phase only : mean TVD = 0.486


wrong both       : mean TVD = 0.975


## Logic-only vs phase-only vs combined

In [4]:
for mode in ['logic', 'phase', 'both']:
    res = obfuscate(qc, mode=mode, seed=42)
    ok = deobfuscate(res.circuit, res.logic_key, res.phase_key)
    print(f'{mode:<6} correct key exact: {functionally_equivalent(ok, qc)}   '
          f'obfuscated-run TVD: {tvd(ref, simulate(res.circuit, seed=1)):.3f}')

logic  correct key exact: True   obfuscated-run TVD: 0.943
phase  correct key exact: True   obfuscated-run TVD: 0.850


both   correct key exact: True   obfuscated-run TVD: 0.959
